# Optimize SCALP-lite Parameters

Select one dataset, run broad PCA latent Bayesian optimization, refine around the good region with GPLVM, and save the best graph parameters for downstream notebooks.


In [11]:
import sys
import subprocess
import warnings

warnings.filterwarnings(
    "ignore",
    message=r"\s*Found Intel OpenMP .* LLVM OpenMP .*",
    category=RuntimeWarning,
    module=r"threadpoolctl",
)


def ensure_bo_dependencies():
    try:
        import botorch  # noqa: F401
        import gpytorch  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", "..[bo]"])


ensure_bo_dependencies()


In [12]:
import numpy as np

from scalp_lite.metrics import score_embedding
from scalp_lite.notebook_utils import dataset_config, load_preprocessed_data, make_estimator, save_optimized_graph_params
from scalp_lite.optimization import latent_bayesopt


selected_dataset = "zebrafish"
dataset = dataset_config(selected_dataset)

# Keep objective evaluations manageable. Increase after validating the search setup.
preprocess_overrides = {"max_cells": 2000}
random_state = 0
estimator = make_estimator(dataset, n_components=100, random_state=random_state)
adata = load_preprocessed_data(estimator, dataset, **preprocess_overrides)
adata


AnnData object with n_obs × n_vars = 2000 × 2000
    obs: 'Stage', 'gt_terminal_states', 'lineages'
    var: 'scalp_lite_hvg_score', 'scalp_lite_selected'
    uns: 'Stage_colors', 'gt_terminal_states_colors', 'lineages_colors'
    obsm: 'X_force_directed', 'X_pca'

In [13]:
base_graph_params = {
    "n_neighbors": 15,
    "intra_fraction": 0.5,
    "n_inter_edges": 2,
    "metric": "euclidean",
    "assignment_quantile": 0.8,
    "hubness_correction": "csls",
    "hubness_k": 10,
    "rank_correction": True,
    "edge_weighting": "binary",
    "mutual_neighbors": False,
    "neighbor_mode": "distance",
    "symmetrize": True,
}

search_space = {
    "n_neighbors": {"type": "int", "bounds": [5, 40]},
    "intra_fraction": {"type": "float", "bounds": [0.2, 0.9]},
    "n_inter_edges": {"type": "int", "bounds": [1, 8]},
    "assignment_quantile": {"type": "float", "bounds": [0.05, 1.0]},
    "hubness_k": {"type": "int", "bounds": [3, 30]},
    "rank_correction": {"type": "categorical", "values": [False, True]},
    "edge_weighting": {"type": "categorical", "values": ["binary", "distance"]},
    "mutual_neighbors": {"type": "categorical", "values": [False, True]},
}

base_graph_params, search_space


({'n_neighbors': 15,
  'intra_fraction': 0.5,
  'n_inter_edges': 2,
  'metric': 'euclidean',
  'assignment_quantile': 0.8,
  'hubness_correction': 'csls',
  'hubness_k': 10,
  'rank_correction': True,
  'edge_weighting': 'binary',
  'mutual_neighbors': False,
  'neighbor_mode': 'distance',
  'symmetrize': True},
 {'n_neighbors': {'type': 'int', 'bounds': [5, 40]},
  'intra_fraction': {'type': 'float', 'bounds': [0.2, 0.9]},
  'n_inter_edges': {'type': 'int', 'bounds': [1, 8]},
  'assignment_quantile': {'type': 'float', 'bounds': [0.05, 1.0]},
  'hubness_k': {'type': 'int', 'bounds': [3, 30]},
  'rank_correction': {'type': 'categorical', 'values': [False, True]},
  'edge_weighting': {'type': 'categorical', 'values': ['binary', 'distance']},
  'mutual_neighbors': {'type': 'categorical', 'values': [False, True]}})

In [14]:
def metric_value(scores, key, default=0.0):
    value = float(scores.get(key, default))
    return default if not np.isfinite(value) else value


def scalp_objective(params):
    graph_params = {**base_graph_params, **params}
    trial = adata.copy()
    try:
        trial.obsm["X_scalp"] = estimator.embed(
            trial,
            **graph_params,
            embedding_random_state=random_state,
            n_epochs=100,
        )
        scores = score_embedding(
            trial,
            embedding_key="X_scalp",
            batch_key=dataset["batch_key"],
            label_key=dataset["label_key"] if dataset["label_key"] in trial.obs else None,
        ).iloc[0]
    except Exception as exc:
        print(f"failed params={graph_params}: {exc}")
        return -1e9

    label_agreement = metric_value(scores, "knn_label_agreement")
    batch_mixing = metric_value(scores, "batch_mixing")
    graph_density = metric_value(scores, "graph_density")
    # Maximize biological coherence and batch mixing; tune weights for your target use case.
    return float(label_agreement + 0.25 * batch_mixing - 0.05 * graph_density)


In [ ]:
# First pass: broad PCA latent BO for a cheap sanity-check search.
bo_result = latent_bayesopt(
    scalp_objective,
    search_space,
    n_initial=12,
    latent_dim=3,
    n_iterations=12,
    embedding_model="pca",
    acquisition="ei",
    batch_size=1,
    random_state=random_state,
    invalid_score=-1e9,
)

bo_result["best_params"], bo_result["best_score"]


In [ ]:
bo_result["history"].sort_values("score", ascending=False).head(10)


,iteration,phase,score,n_neighbors,intra_fraction,n_inter_edges,assignment_quantile,hubness_k,rank_correction,edge_weighting,mutual_neighbors
22,11,bo,0.934225,26,0.649872,4,0.940123,20,True,distance,True
10,0,initial,0.933700,19,0.367559,7,0.882660,5,False,distance,False
5,0,initial,0.930150,32,0.898047,4,0.981794,13,True,distance,True
16,5,bo,0.929475,25,0.621742,5,0.914286,20,True,distance,True
20,9,bo,0.927675,27,0.690449,4,0.917463,20,True,distance,True
23,12,bo,0.927375,26,0.644941,4,0.947363,20,True,distance,True
1,0,initial,0.924450,28,0.624645,8,0.743022,20,True,distance,True
12,1,bo,0.922425,24,0.488499,7,0.814855,20,True,distance,True
14,3,bo,0.920225,25,0.567056,6,0.874488,20,True,distance,True
13,2,bo,0.919450,24,0.536409,6,0.847514,20,True,distance,True


In [ ]:
def compact_bounds(best, name, low, high, radius, *, integer=False):
    center = best[name]
    new_low = max(low, center - radius)
    new_high = min(high, center + radius)
    if integer:
        new_low = int(round(new_low))
        new_high = int(round(new_high))
        if new_low == new_high:
            new_low = max(low, new_low - 1)
            new_high = min(high, new_high + 1)
    return [new_low, new_high]


best_pca_params = bo_result["best_params"]

# Second-stage search ranges, centered on the good PCA solution but still wide enough to move.
compact_search_space = {
    "n_neighbors": {
        "type": "int",
        "bounds": compact_bounds(best_pca_params, "n_neighbors", 5, 40, 8, integer=True),
    },
    "intra_fraction": {
        "type": "float",
        "bounds": compact_bounds(best_pca_params, "intra_fraction", 0.2, 0.9, 0.12),
    },
    "n_inter_edges": {
        "type": "int",
        "bounds": compact_bounds(best_pca_params, "n_inter_edges", 1, 8, 2, integer=True),
    },
    "assignment_quantile": {
        "type": "float",
        "bounds": compact_bounds(best_pca_params, "assignment_quantile", 0.05, 1.0, 0.15),
    },
    "hubness_k": {
        "type": "int",
        "bounds": compact_bounds(best_pca_params, "hubness_k", 3, 30, 6, integer=True),
    },
    # Keep the categorical choices that worked in the PCA stage fixed for the refinement.
    "rank_correction": {"type": "categorical", "values": [best_pca_params["rank_correction"]]},
    "edge_weighting": {"type": "categorical", "values": [best_pca_params["edge_weighting"]]},
    "mutual_neighbors": {"type": "categorical", "values": [best_pca_params["mutual_neighbors"]]},
}

compact_search_space


{'n_neighbors': {'type': 'int', 'bounds': [18, 34]},
 'intra_fraction': {'type': 'float',
  'bounds': [0.529871614435544, 0.769871614435544]},
 'n_inter_edges': {'type': 'int', 'bounds': [2, 6]},
 'assignment_quantile': {'type': 'float', 'bounds': [0.7901229065244357, 1.0]},
 'hubness_k': {'type': 'int', 'bounds': [14, 26]},
 'rank_correction': {'type': 'categorical', 'values': [True]},
 'edge_weighting': {'type': 'categorical', 'values': ['distance']},
 'mutual_neighbors': {'type': 'categorical', 'values': [True]}}

In [ ]:
# Second pass: nonlinear GPLVM latent BO over the compact space around the PCA solution.
gplvm_result = latent_bayesopt(
    scalp_objective,
    compact_search_space,
    n_initial=12,
    latent_dim=3,
    n_iterations=12,
    embedding_model="gplvm",
    acquisition="ei",
    batch_size=1,
    random_state=random_state + 1,
    invalid_score=-1e9,
)

gplvm_result["best_params"], gplvm_result["best_score"]


({'n_neighbors': 34,
  'intra_fraction': 0.6047111629180605,
  'n_inter_edges': 6,
  'assignment_quantile': 0.9638387211776556,
  'hubness_k': 19,
  'rank_correction': True,
  'edge_weighting': 'distance',
  'mutual_neighbors': True},
 0.9334)

In [ ]:
gplvm_result["history"].sort_values("score", ascending=False).head(10)


,iteration,phase,score,n_neighbors,intra_fraction,n_inter_edges,assignment_quantile,hubness_k,rank_correction,edge_weighting,mutual_neighbors
1,0,initial,0.933400,34,0.604711,6,0.963839,19,True,distance,True
10,0,initial,0.930225,31,0.539374,6,0.901062,25,True,distance,True
11,0,initial,0.928300,25,0.544836,4,0.969071,22,True,distance,True
3,0,initial,0.927675,30,0.659026,6,0.955596,18,True,distance,True
19,8,bo,0.927625,27,0.659930,4,0.909063,22,True,distance,True
0,0,initial,0.927400,26,0.757983,4,0.820379,24,True,distance,True
6,0,initial,0.926775,19,0.646317,3,0.995957,23,True,distance,True
18,7,bo,0.925150,27,0.655693,4,0.913583,22,True,distance,True
14,3,bo,0.924825,27,0.652421,4,0.910108,21,True,distance,True
22,11,bo,0.924600,27,0.654506,4,0.915105,22,True,distance,True


In [ ]:
optimization_results = {"pca": bo_result, "gplvm": gplvm_result}
best_model_name, best_result = max(
    optimization_results.items(),
    key=lambda item: item[1]["best_score"],
)

optimized_graph_params = {**base_graph_params, **best_result["best_params"]}
params_path = save_optimized_graph_params(
    selected_dataset,
    optimized_graph_params,
    metadata={
        "best_model": best_model_name,
        "best_score": best_result["best_score"],
        "pca_best_score": bo_result["best_score"],
        "gplvm_best_score": gplvm_result["best_score"],
        "preprocess_overrides": preprocess_overrides,
        "random_state": random_state,
    },
)

params_path, optimized_graph_params


(PosixPath('/Users/fabriziocosta/Resilio Sync/Sync/Projects/ACTIVE/scalp-lite/data/optimized_params/pancreas_graph_params.json'),
 {'n_neighbors': 26,
  'intra_fraction': 0.649871614435544,
  'n_inter_edges': 4,
  'metric': 'euclidean',
  'assignment_quantile': 0.9401229065244358,
  'hubness_correction': 'csls',
  'hubness_k': 20,
  'rank_correction': True,
  'edge_weighting': 'distance',
  'mutual_neighbors': True,
  'neighbor_mode': 'distance',
  'symmetrize': True})